# IDX Exchange DS Internship - Week Three - Data Preprocessing

* Handle missing values (decide whether to drop, impute, or flag).
* Convert categorical fields to numeric (encoding).
* Normalize numerical features if needed.
* Create train/test split: use the most recent month of available data as the test set, and the X months immediately preceding it as the training set. X is not fixed — treat the training window length as a tunable choice and experiment to determine the optimal value of X.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

## Load In Data

In [2]:
# Read in sold: Jan, 2024 - April, 2026

sold_24_1 = pd.read_csv("../csv/CRMLSSold202401_filled.csv")
sold_24_2 = pd.read_csv("../csv/CRMLSSold202402.csv")
sold_24_3 = pd.read_csv("../csv/CRMLSSold202403_filled.csv")
sold_24_4 = pd.read_csv("../csv/CRMLSSold202404_filled.csv", low_memory=False)
sold_24_5 = pd.read_csv("../csv/CRMLSSold202405_filled.csv")
sold_24_6 = pd.read_csv("../csv/CRMLSSold202406_filled.csv")
sold_24_7 = pd.read_csv("../csv/CRMLSSold202407_filled.csv")
sold_24_8 = pd.read_csv("../csv/CRMLSSold202408.csv")
sold_24_9 = pd.read_csv("../csv/CRMLSSold202409.csv")
sold_24_10 = pd.read_csv("../csv/CRMLSSold202410.csv")
sold_24_11 = pd.read_csv("../csv/CRMLSSold202411.csv")
sold_24_12 = pd.read_csv("../csv/CRMLSSold202412.csv")
sold_25_1 = pd.read_csv("../csv/CRMLSSold202501_filled.csv")
sold_25_2 = pd.read_csv("../csv/CRMLSSold202502.csv")
sold_25_3 = pd.read_csv("../csv/CRMLSSold202503.csv")
sold_25_4 = pd.read_csv("../csv/CRMLSSold202504.csv")
sold_25_5 = pd.read_csv("../csv/CRMLSSold202505.csv")
sold_25_6 = pd.read_csv("../csv/CRMLSSold202506.csv", low_memory=False)
sold_25_7 = pd.read_csv("../csv/CRMLSSold202507.csv")
sold_25_8 = pd.read_csv("../csv/CRMLSSold202508.csv")
sold_25_9 = pd.read_csv("../csv/CRMLSSold202509.csv")
sold_25_10 = pd.read_csv("../csv/CRMLSSold202510.csv")
sold_25_11 = pd.read_csv("../csv/CRMLSSold202511.csv")
sold_25_12 = pd.read_csv("../csv/CRMLSSold202512.csv")
sold_26_1 = pd.read_csv("../csv/CRMLSSold202601.csv", low_memory=False)
sold_26_2 = pd.read_csv("../csv/CRMLSSold202602.csv")
sold_26_3 = pd.read_csv("../csv/CRMLSSold202603.csv")
sold_26_4 = pd.read_csv("../csv/CRMLSSold202604.csv")

In [3]:
# Set which datasets are reserved for testing
# Use the most recent month, April 2026, for now

sold_26_4["TestSetTrue"] = 1

In [4]:
# Combine into one dataframe

sold_list = [sold_24_1, sold_24_2, sold_24_3, sold_24_4, sold_24_5, sold_24_6,
                 sold_24_7, sold_24_8, sold_24_9, sold_24_10, sold_24_11, sold_24_12,
                 sold_25_1, sold_25_2, sold_25_3, sold_25_4, sold_25_5, sold_25_6,
                 sold_25_7, sold_25_8, sold_25_9, sold_25_10, sold_25_11, sold_25_12, 
                 sold_26_1, sold_26_2, sold_26_3, sold_26_4]
sold = pd.concat(sold_list)

In [5]:
# Filter on PropertyType == Residential and PropertySubType == SingleFamilyResidence

filtered_sold = sold[(sold["PropertyType"] == "Residential") & (sold["PropertySubType"] == "SingleFamilyResidence")]

In [6]:
filtered_sold.head()

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR,OriginatingSystemName,OriginatingSystemSubName,TestSetTrue
5,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,...,NaN,NaN,NaN,True,True,NaN,NaN,NaN,NaN,NaN
9,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,...,NaN,NaN,NaN,True,True,NaN,NaN,NaN,NaN,NaN
28,NaN,True,NaN,NaN,False,2100000.0,1067652762,jenniferarielkennedy@gmail.com,2024-01-02,2100000.0,...,0.0,11219.0,NaN,False,False,NaN,NaN,NaN,NaN,NaN
29,Tile,True,NaN,NaN,False,1950000.0,1063453216,kira@mendosir.com,2024-01-22,1950000.0,...,NaN,74487.6,NaN,False,False,NaN,NaN,NaN,NaN,NaN
31,"Tile,Wood",True,NaN,NaN,NaN,NaN,1061988701,dannyduong@gmail.com,2024-01-31,2340000.0,...,NaN,8712.0,NaN,True,True,NaN,NaN,NaN,NaN,NaN


## Remove Impossibilities

In [246]:
# Note original shape before any processing

original = filtered_sold.copy()
original.shape

(309735, 85)

In [247]:
# Remove duplicates, representing as any rows with the same ListingKey and CloseDate

filtered_sold = filtered_sold.drop_duplicates(subset=["ListingKey", "CloseDate"])

filtered_sold.shape[0]

309706

In [248]:
# Remove items with no ClosePrice or a ClosePrice <= 0

filtered_sold = filtered_sold[filtered_sold["ClosePrice"] > 0]
filtered_sold.shape

(309703, 85)

In [249]:
# Drop columns with a ListDate that comes after a CloseDate

filtered_sold = filtered_sold[~(filtered_sold["CloseDate"] < filtered_sold["ListingContractDate"])]
filtered_sold.shape

(309659, 85)

In [250]:
# Remove impossible square footage (but keep missing values for now)

filtered_sold = filtered_sold[~(filtered_sold["LivingArea"] <= 0)]
print(filtered_sold.shape)

filtered_sold = filtered_sold[~(filtered_sold["LotSizeSquareFeet"] <= 0)]
print(filtered_sold.shape)

#filtered_sold = filtered_sold[~(filtered_sold["LotSizeAcres"] <= 0)]
#print(filtered_sold.shape)

#filtered_sold = filtered_sold[~(filtered_sold["LotSizeArea"] <= 0)]
#print(filtered_sold.shape)

### Notes:
# LotSizeAcres has a lot of missing values. LotSizeSquareFeet and LotSizeArea seem to be very similar
# Might be best to only use LivingArea and LotSizeSquareFeet for measurements

(309533, 85)
(309371, 85)


In [251]:
# Remove inconsistent values, like a very small living space but more than 1 bedroom or 1 bathroom

filtered_sold = filtered_sold[~((filtered_sold["LivingArea"] <= 200) & (filtered_sold["BathroomsTotalInteger"] > 1))]
filtered_sold = filtered_sold[~((filtered_sold["LivingArea"] <= 200) & (filtered_sold["BedroomsTotal"] > 1))]

filtered_sold.shape

(309366, 85)

In [252]:
number_dropped = original.shape[0] - filtered_sold.shape[0]
percent_dropped = number_dropped/original.shape[0] * 100

print(f"Dropped {number_dropped} rows, {round(percent_dropped, 2)}%")

Dropped 369 rows, 0.12%


## Drop Large Missing Values

In [253]:
# Check which features have the highest amount of missing values

na_check = filtered_sold.isnull().sum().sort_values(ascending=False)
na_check

FireplacesTotal           309366
CoveredSpaces             309366
AboveGradeFinishedArea    309366
TaxAnnualAmount           309366
TaxYear                   309366
                           ...  
PropertySubType                0
ListingContractDate            0
MlsStatus                      0
CountyOrParish                 0
ListOfficeName                 0
Length: 85, dtype: int64

In [254]:
# These columns have more than half of their rows missing

num_rows = filtered_sold.shape[0]
half = num_rows/2
print(half)

half_na = na_check[na_check >= half]
half_na.sort_values(ascending=False)

154683.0


FireplacesTotal                 309366
CoveredSpaces                   309366
AboveGradeFinishedArea          309366
TaxAnnualAmount                 309366
TaxYear                         309366
ElementarySchoolDistrict        309366
BusinessType                    309366
MiddleOrJuniorSchoolDistrict    309366
WaterfrontYN                    309231
BelowGradeFinishedArea          307464
BasementYN                      301915
TestSetTrue                     296895
BuilderName                     294922
LotSizeDimensions               290472
BuildingAreaTotal               289848
CoBuyerAgentFirstName           281586
OriginatingSystemName           276394
OriginatingSystemSubName        276394
BuyerAgencyCompensation         267163
BuyerAgencyCompensationType     267149
ElementarySchool                267007
MiddleOrJuniorSchool            266678
HighSchool                      253965
CoListAgentFirstName            238836
CoListAgentLastName             238629
CoListOfficeName         

In [255]:
# Drop columns that have more than half of their values missing

# Make sure the keep the test set column
keep_test_set = filtered_sold["TestSetTrue"]

cleaned_na = filtered_sold.drop(columns = half_na.index)
cleaned_na["TestSetTrue"] = keep_test_set
number_dropped = filtered_sold.shape[1] - cleaned_na.shape[1]
percent_dropped = number_dropped/filtered_sold.shape[1] * 100

print(cleaned_na.shape)
print(f"Dropped {number_dropped} columns, {round(percent_dropped, 2)}%")

(309366, 56)
Dropped 29 columns, 34.12%


## Remove Leakage

In [256]:
cleaned_na.columns

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice',
       'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice',
       'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude',
       'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice',
       'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'BuyerOfficeAOR', 'YearBuilt',
       'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City',
       'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
  

In [257]:
# Some columns should be deleted because they correlate too highly with ClosePrice
cleaned_leak = cleaned_na.drop(columns = ["OriginalListPrice", "ListPrice"])

# Some columns should be deleted because they are only known after a sale has been finalized
# Drop anything related to buyer or the final purchase
# Keeping the target variable "ClosePrice" for removal later
cleaned_leak = cleaned_leak.drop(columns = ["DaysOnMarket", "BuyerOfficeName", "BuyerAgentMlsId",
                                            "BuyerAgentFirstName", "BuyerAgentLastName", "MlsStatus", "BuyerOfficeAOR",
                                            "PurchaseContractDate", "BuyerAgentAOR", "ContractStatusChangeDate", 
                                            "ListingContractDate"])

In [258]:
number_dropped = cleaned_na.shape[1] - cleaned_leak.shape[1]
percent_dropped = number_dropped/cleaned_na.shape[1] * 100

print(cleaned_leak.shape)
print(f"Dropped {number_dropped} columns, {round(percent_dropped, 2)}%")

(309366, 43)
Dropped 13 columns, 23.21%


## Handle Remaining Missing Values

In [259]:
# Don't look at columns with no missing values right now

zero_check = cleaned_leak.isna().sum().sort_values(ascending=False)
zero_check = zero_check[zero_check == 0]
zero_check

ListingKey           0
StateOrProvince      0
BedroomsTotal        0
ListingId            0
CountyOrParish       0
ListingKeyNumeric    0
ListOfficeName       0
PropertyType         0
ClosePrice           0
CloseDate            0
PropertySubType      0
dtype: int64

In [260]:
# Need to fill the null values in these columns or remove them

investigate = cleaned_leak.drop(columns = zero_check.index)
investigate.isnull().mean().sort_values(ascending=False) * 100

TestSetTrue              95.968852
MainLevelBedrooms        40.233251
Flooring                 35.734696
AssociationFee           30.196596
HighSchoolDistrict       25.995100
MLSAreaMajor             14.218111
ListAgentAOR             13.657933
Stories                  12.708248
AttachedGarageYN         11.606964
PoolPrivateYN             9.617734
Levels                    9.449972
ViewYN                    8.917593
NewConstructionYN         7.244170
GarageSpaces              3.691097
LotSizeAcres              1.720616
LotSizeSquareFeet         1.718353
LotSizeArea               1.710595
Longitude                 0.875339
Latitude                  0.875339
ListAgentFirstName        0.801639
ListAgentEmail            0.238229
StreetNumberNumeric       0.117014
ParkingTotal              0.112488
UnparsedAddress           0.095033
City                      0.078548
FireplaceYN               0.065618
YearBuilt                 0.064648
LivingArea                0.053658
ListAgentFullName   

#### ListAgent Information

In [261]:
# ListAgentFirstName with ListAgentLastName encode pretty much the same data as ListAgentFullName

cleaned_further = cleaned_leak.drop(columns = ["ListAgentFirstName", "ListAgentLastName", "ListAgentAOR", "ListAgentEmail", "ListAgentFullName"])

In [262]:
number_dropped = cleaned_leak.shape[1] - cleaned_further.shape[1]
percent_dropped = number_dropped/cleaned_leak.shape[1] * 100

prev = cleaned_further.shape
print(prev)
print(f"Dropped {number_dropped} columns, {round(percent_dropped, 2)}%")

(309366, 38)
Dropped 5 columns, 11.63%


#### Levels vs. Stories

In [263]:
# Stories and Levels seem to encode similar data. Check the difference

print(cleaned_further["Levels"].value_counts())
print("-----")
print(cleaned_further["Stories"].value_counts())

Levels
One                               173633
Two                                94409
ThreeOrMore                         5818
MultiSplit                          4300
One,Two                              657
Two,MultiSplit                       459
Two,One                              264
ThreeOrMore,MultiSplit               208
One,MultiSplit                       170
Two,ThreeOrMore                      107
One,ThreeOrMore                       47
MultiSplit,One                        26
One,Two,MultiSplit                    10
Two,ThreeOrMore,MultiSplit             7
One,Two,ThreeOrMore                    6
ThreeOrMore,One                        6
One,Two,ThreeOrMore,MultiSplit         3
Two,MultiSplit,One                     1
Name: count, dtype: int64
-----
Stories
1.0    175259
2.0     94792
Name: count, dtype: int64


In [264]:
# Check how Stories and Levels interact

cleaned_further[(cleaned_further["Levels"] != "One") & (cleaned_further["Levels"] != "Two")][["Stories", "Levels"]].value_counts()

Stories  Levels                        
1.0      One,Two                           657
2.0      Two,MultiSplit                    459
1.0      Two,One                           264
         One,MultiSplit                    170
2.0      Two,ThreeOrMore                   107
1.0      One,ThreeOrMore                    47
         MultiSplit,One                     26
         MultiSplit                         13
         One,Two,MultiSplit                 10
2.0      Two,ThreeOrMore,MultiSplit          7
1.0      One,Two,ThreeOrMore                 6
         ThreeOrMore,One                     6
         One,Two,ThreeOrMore,MultiSplit      3
         ThreeOrMore                         1
         Two,MultiSplit,One                  1
Name: count, dtype: int64

In [265]:
# What values in Levels are showing up as Null in Stories

cleaned_further[cleaned_further["Stories"].isna() == True]["Levels"].value_counts(dropna=False)

Levels
NaN                       29003
ThreeOrMore                5817
MultiSplit                 4287
ThreeOrMore,MultiSplit      208
Name: count, dtype: int64

In [266]:
# Impute Stories by filling in these null values.
# Multisplit means a loft, so represent as a half value.

cleaned_further["imputed_stories"] = cleaned_further["Stories"]

cleaned_further.loc[(cleaned_further["imputed_stories"].isna()) & (cleaned_further["Levels"] == "ThreeOrMore"), "imputed_stories"] = 3.0
cleaned_further.loc[(cleaned_further["imputed_stories"].isna()) & (cleaned_further["Levels"] == "MultiSplit"), "imputed_stories"] = 1.5
cleaned_further.loc[(cleaned_further["imputed_stories"].isna()) & (cleaned_further["Levels"] == "ThreeOrMore,MultiSplit"), "imputed_stories"]= 3.5

cleaned_further["imputed_stories"].value_counts(dropna=False)

imputed_stories
1.0    175259
2.0     94792
NaN     29003
3.0      5817
1.5      4287
3.5       208
Name: count, dtype: int64

In [267]:
# Calculate LivingArea median for each story option

one_story_median = cleaned_further[cleaned_further["imputed_stories"] == 1.0 ]["LivingArea"].median()
one_half_story_median = cleaned_further[cleaned_further["imputed_stories"] == 1.5 ]["LivingArea"].median()
two_story_median = cleaned_further[cleaned_further["imputed_stories"] == 2.0 ]["LivingArea"].median()
three_story_median = cleaned_further[cleaned_further["imputed_stories"] == 3.0 ]["LivingArea"].median()
three_half_story_median = cleaned_further[cleaned_further["imputed_stories"] == 3.5 ]["LivingArea"].median()

print(one_story_median, one_half_story_median, two_story_median, three_story_median, three_half_story_median)

1536.0 2286.5 2380.0 2614.0 2830.0


In [268]:
# Impute remaining missing stories based on LivingArea

midpoint_1 = (one_story_median + one_half_story_median) / 2
midpoint_2 = (one_half_story_median + two_story_median) / 2
midpoint_3 = (two_story_median + three_story_median) / 2
midpoint_4 = (three_story_median + three_half_story_median) / 2

# [-, midpoint_1) = 1 story
# [midpoint_1, midpoint_2) = 1.5 stories
# [midpoint_2, midpoint_3) = 2 stories
# [midpoint_3, minpoint_4) = 3 stories
# [midpoint_4, +] = 3.5 stories

cleaned_further.loc[(cleaned_further["imputed_stories"].isna()) & 
                    (cleaned_further["LivingArea"] < midpoint_1),
                    "imputed_stories"] = 1.0

cleaned_further.loc[(cleaned_further["imputed_stories"].isna()) & 
                    (cleaned_further["LivingArea"] >= midpoint_1) &
                    (cleaned_further["LivingArea"] < midpoint_2),
                    "imputed_stories"] = 1.5

cleaned_further.loc[(cleaned_further["imputed_stories"].isna()) & 
                    (cleaned_further["LivingArea"] >= midpoint_2) &
                    (cleaned_further["LivingArea"] < midpoint_3),
                    "imputed_stories"]= 2.0

cleaned_further.loc[(cleaned_further["imputed_stories"].isna()) & 
                    (cleaned_further["LivingArea"] >= midpoint_3) &
                    (cleaned_further["LivingArea"] < midpoint_4),
                    "imputed_stories"] = 3.0

cleaned_further.loc[(cleaned_further["imputed_stories"].isna()) & 
                    (cleaned_further["LivingArea"] >= midpoint_4),
                    "imputed_stories"] = 3.5

In [269]:
# Drop these two columns as their information is encoded in "Imputed Stories" now

cleaned_further = cleaned_further.drop(columns = ["Stories", "Levels"])

In [270]:
number_dropped = prev[1] - cleaned_further.shape[1]
percent_dropped = number_dropped/prev[1] * 100

prev = cleaned_further.shape
print(prev)
print(f"Dropped {number_dropped} columns, {round(percent_dropped, 2)}%")

(309366, 37)
Dropped 1 columns, 2.63%


#### BathroomsTotalInteger

In [271]:
# Get median bathrooms from each story

one_story_bath = cleaned_further[cleaned_further["imputed_stories"] == 1.0 ]["BathroomsTotalInteger"].median()
one_half_story_bath = cleaned_further[cleaned_further["imputed_stories"] == 1.5 ]["BathroomsTotalInteger"].median()
two_story_bath = cleaned_further[cleaned_further["imputed_stories"] == 2.0 ]["BathroomsTotalInteger"].median()
three_story_bath = cleaned_further[cleaned_further["imputed_stories"] == 3.0 ]["BathroomsTotalInteger"].median()
three_half_story_bath = cleaned_further[cleaned_further["imputed_stories"] == 3.5 ]["BathroomsTotalInteger"].median()

print(one_story_bath, one_half_story_bath, two_story_bath, three_story_bath, three_half_story_bath)

2.0 3.0 3.0 3.0 4.0


In [272]:
# Impute BathroomsTotalInteger with median bathrooms of each story

cleaned_further.loc[(cleaned_further["BathroomsTotalInteger"].isna()) &
                    (cleaned_further["imputed_stories"] == 1.0),
                    "BathroomsTotalInteger"] = one_story_bath

cleaned_further.loc[(cleaned_further["BathroomsTotalInteger"].isna()) &
                    (cleaned_further["imputed_stories"] == 1.5),
                    "BathroomsTotalInteger"] = one_half_story_bath

cleaned_further.loc[(cleaned_further["BathroomsTotalInteger"].isna()) &
                    (cleaned_further["imputed_stories"] == 2.0),
                    "BathroomsTotalInteger"] = two_story_bath

cleaned_further.loc[(cleaned_further["BathroomsTotalInteger"].isna()) &
                    (cleaned_further["imputed_stories"] == 3.0),
                    "BathroomsTotalInteger"] = three_story_bath

cleaned_further.loc[(cleaned_further["BathroomsTotalInteger"].isna()) &
                    (cleaned_further["imputed_stories"] == 3.5),
                    "BathroomsTotalInteger"] = three_half_story_bath

#### Association Fee

In [273]:
# Impute AssociationFee with median
# Median is 0

cleaned_further["AssociationFee"] = cleaned_further["AssociationFee"].fillna(cleaned_further["AssociationFee"].median())

#### Flooring

In [274]:
# Fill null of Flooring with "Unknown"

cleaned_further["Flooring"] = cleaned_further["Flooring"].fillna("Unknown")

#### LotSize Units

In [275]:
# Check differences in units

### Notes
# LotSizeArea uses ambigious units, seemingly a combination of LotSizeAcres or LotSizeSquareFeet, not super good to use
# This leaves LotSizeAcres and LotSizeSquareFeet. LotSizeSquareFeet appears better since its values are more precise

cleaned_further[["LotSizeAcres", "LotSizeArea", "LotSizeSquareFeet"]]

,LotSizeAcres,LotSizeArea,LotSizeSquareFeet
5,NaN,NaN,NaN
9,NaN,NaN,NaN
28,0.2576,11219.00,11219.0
29,1.7100,1.71,74487.6
31,0.2000,8712.00,8712.0
...,...,...,...
24227,0.1830,7971.00,7971.0
24228,0.1194,5200.00,5200.0
24229,0.2300,10019.00,10019.0
24231,0.2777,12096.00,12096.0


In [276]:
# Since LotSizeSquareFeet is the best LotSize column, investigate if other features can be used to fill its null values
# Where LotSizeSquareFeet is null and the other values exist, both values seem to be in square feet

cleaned_further[(cleaned_further["LotSizeSquareFeet"].isna() == True) &
                (cleaned_further["LotSizeAcres"].isna() == False) &
                (cleaned_further["LotSizeArea"].isna() == False)][["LotSizeAcres", "LotSizeArea"]]

,LotSizeAcres,LotSizeArea
12995,57092.0000,57092.0000
6915,64013.0000,64013.0000
17230,0.0000,0.0000
11962,86248.0000,86248.0000
14829,87991.0000,87991.0000
17828,571837.0000,571837.0000
19452,81282.0000,81282.0000
12973,67790.1556,67790.1556
20535,54450.0000,54450.0000
8769,63598.0000,63598.0000


In [277]:
# Fill missing LotSizeSquareFeet values with LotSizeArea

cleaned_further["LotSizeSquareFeet"] = cleaned_further["LotSizeSquareFeet"].fillna(cleaned_further["LotSizeArea"])

In [278]:
# Having multiple LotSize columns in redundant. Keep LotSizeSquareFeet and drop LotSizeAcres and LotSizeArea

cleaned_further = cleaned_further.drop(columns = ["LotSizeAcres", "LotSizeArea"])

In [279]:
number_dropped = prev[1] - cleaned_further.shape[1]
percent_dropped = number_dropped/prev[1] * 100

prev = cleaned_further.shape
print(prev)
print(f"Dropped {number_dropped} columns, {round(percent_dropped, 2)}%")

(309366, 35)
Dropped 2 columns, 5.41%


#### MainLevelBedrooms

In [280]:
# Would expect difference to all be postive (BedroomsTotal should be greater than MainLevelBedrooms)
# Should negative values be deleted for being impossible?

bedroom_diff = cleaned_further["BedroomsTotal"] - cleaned_further["MainLevelBedrooms"]
bedroom_diff.value_counts().sort_index()

-3330.0         1
-440.0          1
-347.0          1
-43.0           1
-40.0           1
-39.0           2
-38.0           1
-30.0          11
-29.0          12
-28.0           4
-20.0           4
-19.0           4
-18.0           1
-16.0           1
-10.0           4
-9.0            1
-8.0            7
-7.0            4
-6.0            4
-5.0            4
-4.0            2
-3.0           21
-2.0          147
-1.0         1631
 0.0       103874
 1.0        10838
 2.0        13027
 3.0        31588
 4.0        21013
 5.0         2363
 6.0          254
 7.0           40
 8.0           17
 9.0            4
 10.0           1
 11.0           3
 12.0           3
 13.0           2
 34.0           1
Name: count, dtype: int64

In [281]:
# If only one story, impute MainLevelBedrooms as same as BedroomsTotal.
# Otherwise, impute average of non one story properties

# If there is one story, fill MainLevelBedrooms with BedroomsTotal.
cleaned_further["MainLevelBedrooms"] = np.where((cleaned_further["imputed_stories"] == 1.0) &
                                                cleaned_further["MainLevelBedrooms"].isna() == True,
                                                cleaned_further["BedroomsTotal"], cleaned_further["MainLevelBedrooms"])

plus_one_story_bed_median = cleaned_further[cleaned_further["imputed_stories"] > 1.0]["MainLevelBedrooms"].median()
cleaned_further["MainLevelBedrooms"] = cleaned_further["MainLevelBedrooms"].fillna(plus_one_story_bed_median)

#### Missing Booleans

In [282]:
# Simply flag as "unknown" for all missing boolean features

cleaned_further["AttachedGarageYN"] = cleaned_further["AttachedGarageYN"].fillna("Unknown")
cleaned_further["PoolPrivateYN"] = cleaned_further["PoolPrivateYN"].fillna("Unknown")
cleaned_further["ViewYN"] = cleaned_further["ViewYN"].fillna("Unknown")
cleaned_further["NewConstructionYN"] = cleaned_further["NewConstructionYN"].fillna("Unknown")
cleaned_further["FireplaceYN"] = cleaned_further["FireplaceYN"].fillna("Unknown")

#### LotSizeSquareFeet and LivingArea

In [283]:
# Calculate the median LivingArea and LotSizeSquareFeet for each number of stories

one_story_living = cleaned_further[cleaned_further["imputed_stories"] == 1.0]["LivingArea"].median()
one_story_feet = cleaned_further[cleaned_further["imputed_stories"] == 1.0]["LotSizeSquareFeet"].median()

half_story_living = cleaned_further[cleaned_further["imputed_stories"] == 1.5]["LivingArea"].median()
half_story_feet = cleaned_further[cleaned_further["imputed_stories"] == 1.5]["LotSizeSquareFeet"].median()

two_story_living = cleaned_further[cleaned_further["imputed_stories"] == 2.0]["LivingArea"].median()
two_story_feet = cleaned_further[cleaned_further["imputed_stories"] == 2.0]["LotSizeSquareFeet"].median()

three_story_living = cleaned_further[cleaned_further["imputed_stories"] == 3.0]["LivingArea"].median()
three_story_feet = cleaned_further[cleaned_further["imputed_stories"] == 3.0]["LotSizeSquareFeet"].median()

three_half_story_living = cleaned_further[cleaned_further["imputed_stories"] == 3.5]["LivingArea"].median()
three_half_story_feet = cleaned_further[cleaned_further["imputed_stories"] == 3.5]["LotSizeSquareFeet"].median()

In [284]:
# Impute LivingArea from previous medians

cleaned_further.loc[(cleaned_further["LivingArea"].isna()) & 
                    (cleaned_further["imputed_stories"] == 1.0),
                    "LivingArea"] = one_story_living

cleaned_further.loc[(cleaned_further["LivingArea"].isna()) & 
                    (cleaned_further["imputed_stories"] == 1.5),
                    "LivingArea"] = half_story_living

cleaned_further.loc[(cleaned_further["LivingArea"].isna()) & 
                    (cleaned_further["imputed_stories"] == 2.0),
                    "LivingArea"] = two_story_living

cleaned_further.loc[(cleaned_further["LivingArea"].isna()) & 
                    (cleaned_further["imputed_stories"] == 3.0),
                    "LivingArea"] = three_story_living

cleaned_further.loc[(cleaned_further["LivingArea"].isna()) & 
                    (cleaned_further["imputed_stories"] == 3.5),
                    "LivingArea"] = three_half_story_living

In [285]:
# Impute LotSizeSquareFeet from previous medians

cleaned_further.loc[(cleaned_further["LotSizeSquareFeet"].isna()) & 
                    (cleaned_further["imputed_stories"] == 1.0),
                    "LotSizeSquareFeet"] = one_story_feet

cleaned_further.loc[(cleaned_further["LotSizeSquareFeet"].isna()) & 
                    (cleaned_further["imputed_stories"] == 1.5),
                    "LotSizeSquareFeet"] = half_story_feet

cleaned_further.loc[(cleaned_further["LotSizeSquareFeet"].isna()) & 
                    (cleaned_further["imputed_stories"] == 2.0),
                    "LotSizeSquareFeet"] = two_story_feet

cleaned_further.loc[(cleaned_further["LotSizeSquareFeet"].isna()) & 
                    (cleaned_further["imputed_stories"] == 3.0),
                    "LotSizeSquareFeet"] = three_story_feet

cleaned_further.loc[(cleaned_further["LotSizeSquareFeet"].isna()) & 
                    (cleaned_further["imputed_stories"] == 3.5),
                    "LotSizeSquareFeet"] = three_half_story_feet

#### Parking

In [286]:
# These have slightly different meanings! Parking spaces in garage vs. parking anywhere

cleaned_further[["GarageSpaces", "ParkingTotal"]]

,GarageSpaces,ParkingTotal
5,2.0,2.0
9,2.0,2.0
28,3.0,3.0
29,2.0,2.0
31,3.0,3.0
...,...,...
24227,2.0,2.0
24228,2.0,2.0
24229,0.0,4.0
24231,0.0,0.0


In [287]:
# This doesn't make much sense. 0 parking spaces but space in the garage.
# Maybe delete positive values for being impossible?

cleaned_further[cleaned_further["ParkingTotal"] == 0]["GarageSpaces"].value_counts()

GarageSpaces
0.0    15575
2.0      374
3.0      100
1.0       50
4.0       12
6.0        4
5.0        3
Name: count, dtype: int64

In [288]:
# For all the ParkingTotal spaces that are null, the GarageSpaces column has values

cleaned_further[cleaned_further["ParkingTotal"].isna() == True]["GarageSpaces"].value_counts(dropna=False)

GarageSpaces
2.0    229
1.0     44
0.0     38
3.0     29
4.0      6
5.0      2
Name: count, dtype: int64

In [289]:
# Impute missing ParkingTotal values with GarageSpaces

cleaned_further["ParkingTotal"] = cleaned_further["ParkingTotal"].fillna(cleaned_further["GarageSpaces"])

In [290]:
# Now, impute values for GarageSpaces by using AttachedGarageYN.
# AttachedGarageYN = false does not necessarily mean no garage exists, just that it's not attached.
# Take median GarageSpaces for each value of AttachedGarageYN

print(cleaned_further[cleaned_further["AttachedGarageYN"] == True]["GarageSpaces"].median())
print(cleaned_further[cleaned_further["AttachedGarageYN"] == False]["GarageSpaces"].median())
print(cleaned_further[cleaned_further["AttachedGarageYN"] == "Unknown"]["GarageSpaces"].median())

2.0
2.0
0.0


In [291]:
# Impute GarageSpaces with the medians created using AttachedGarageYN

cleaned_further.loc[(cleaned_further["GarageSpaces"].isna()) & (cleaned_further["AttachedGarageYN"] == True), "GarageSpaces"] = 2.0
cleaned_further.loc[(cleaned_further["GarageSpaces"].isna()) & (cleaned_further["AttachedGarageYN"] == False), "GarageSpaces"] = 2.0
cleaned_further.loc[(cleaned_further["GarageSpaces"].isna()) & (cleaned_further["AttachedGarageYN"] == "Unknown"), "GarageSpaces"]= 0.0

#### Longitude and Latitude

In [292]:
!pip install uszipcode -q

In [293]:
pip install --upgrade uszipcode -q

Note: you may need to restart the kernel to use updated packages.


In [294]:
# import uszipcode database (above pip installs are necessary for using it)
# website: https://pypi.org/project/uszipcode/
# Has information on zipcodes
# Based on 2020 census data, however it's not out of date, since I'm only using the locational, not demographic data

from uszipcode import SearchEngine
search = SearchEngine()

/opt/anaconda3/lib/python3.13/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [295]:
# Lat-long is either both filled or both empty.

cleaned_further[["Longitude", "Latitude"]].isnull().value_counts()

Longitude  Latitude
False      False       306658
True       True          2708
Name: count, dtype: int64

In [296]:
# Only one missing PostalCode row, drop it

print(cleaned_further["PostalCode"].isnull().sum())
cleaned_further = cleaned_further.dropna(subset = ["PostalCode"])

1


In [297]:
# Goal is to impute longitude and latitude.
# Take all rows where longitude/latitude is missing, get the PostalCode.
# Feed PostalCode into uszipcode search, which will return the PostalCode's longitude, latitude, and city

zipcodes = cleaned_further[cleaned_further["Longitude"].isna() == True][["PostalCode"]].values.flatten().tolist()
result = search.by_zipcode(zipcodes[0])

print(zipcodes[0])
print(result.lat, result.lng, result.major_city)

94560
37.51 -122.04 Newark


In [298]:
# For each postalcode, create a dictionary where the postalcode's long/lat are saved
# Also save PostalCodes that uszipcode marks as invalid, in order to investigate

invalid = []

lat_dict = {}
long_dict = {}

for i in zipcodes:
    result = search.by_zipcode(i)
    if result is None:
        invalid.append(i)
    if result is not None:
        lat_dict[i] = result.lat
        long_dict[i] = result.lng

In [299]:
# Verify that the dictionaries work (match to previous output above)

lat_dict[zipcodes[0]], long_dict[zipcodes[0]]

(37.51, -122.04)

In [300]:
# Check which PostalCodes were marked as invalid

print(len(invalid))
print(invalid)

69
['94611-4007', '94804-1101', '94560-1710', '94582-5069', '95337-6695', '94611-3402', '94710-2809', '95206-2027', '94519-1610', '94609-1626', '94605-3813', '94587-4913', '94588-3111', '95306-9708', '94587-4748', '94501-0000', '94611-3819', '94506-6020', '95336-5956', '95386-9479', '94044-4140', '94549-2925', '94509-6864', '94603-4111', '94536-4263', '95386-9551', '91748-3992', '94530-2567', '94568-4697', '94561-9999', '94509-5639', '94520-1605', '94803-3627', '94513-4478', '94804-4303', '94509-5966', '94513-5168', '94513-4255', '94506-4637', '94513-6479', '94536-5026', '94601-5102', '94590-2400', '95330-8896', '95240-0000', '95330-0000', '94603-2034', '94624-3118', '94505-9480', '94506-1918', '94561-0000', '94601-3843', '94509-8923', '95758-7921', '94531-9317', '95330-0000', '94806-2168', '95125-1529', '94563-3744', '93567', '93350', '94512-4814', '49603', '94541-7165', '94566-2931', '94607-2928', '94578-1907', '94555-3', '94705-1628']


In [301]:
# Appears that there are many ranges of postal codes in the data. Remove these as being impossible values
pattern = r'-'
cleaned_further = cleaned_further[~cleaned_further["PostalCode"].str.contains(pattern, na=False, regex=True)]

# Additionally, there are three normal looking postal codes: 93567, 93350, 49603
# Googling these postal codes reveals that they are not uszipcodes and must be data entry errors. Remove them.
cleaned_further = cleaned_further[~cleaned_further["PostalCode"].str.contains("93567", na=False)]
cleaned_further = cleaned_further[~cleaned_further["PostalCode"].str.contains("93350", na=False)]
cleaned_further = cleaned_further[~cleaned_further["PostalCode"].str.contains("49603", na=False)]

In [302]:
# Create new columns, imputed_long and imputed_lat
# Fill them with the existing long/lat from MLS data
# Then fill remaning null with the dictionaries created from uszipcodes

cleaned_further["imputed_long"] = cleaned_further["Longitude"]
cleaned_further["imputed_lat"] = cleaned_further["Latitude"]

cleaned_further["imputed_long"] = cleaned_further["imputed_long"].fillna(cleaned_further["PostalCode"].map(long_dict))
cleaned_further["imputed_lat"] = cleaned_further["imputed_lat"].fillna(cleaned_further["PostalCode"].map(lat_dict))

In [303]:
# Verify that this is working as intended 

cleaned_further[cleaned_further["PostalCode"] == zipcodes[0]][["PostalCode", "Longitude", "Latitude",
                                                               "imputed_long", "imputed_lat"]].head()

,PostalCode,Longitude,Latitude,imputed_long,imputed_lat
1356,94560,-122.045107,37.555768,-122.045107,37.555768
1413,94560,NaN,NaN,-122.040000,37.510000
2655,94560,-122.035955,37.557666,-122.035955,37.557666
3843,94560,-122.039186,37.559952,-122.039186,37.559952
15005,94560,-122.057013,37.534886,-122.057013,37.534886


In [304]:
number_dropped = prev[0] - cleaned_further.shape[0]
percent_dropped = number_dropped/prev[0] * 100

prev = cleaned_further.shape
print(prev)
print(f"Dropped {number_dropped} rows, {round(percent_dropped, 2)}%")

(307656, 37)
Dropped 1710 rows, 0.55%


#### City

In [305]:
# Do the same process as above, but this time for city.
# For every missing city value, get its assocaited PostalCode and pass that through uszipcode.

city_zipcodes = cleaned_further[cleaned_further["City"].isna() == True][["PostalCode"]].values.flatten().tolist()

city_invalid = []
city_dict = {}

for i in city_zipcodes:
    result = search.by_zipcode(i)
    if result is None:
        city_invalid.append(i)
    if result is not None:
        city_dict[i] = result.major_city

In [306]:
# Check which zipcodes uszipcode marked as invalid

print(len(city_invalid))
print(city_invalid)

3
['92440', '92440', '93506']


In [307]:
# A google search verifies that 92440 and 93506 are not valid us zipcodes.

# Delete rows with PostalCode 93506, since there are very few of them.
cleaned_further = cleaned_further[~cleaned_further["PostalCode"].str.contains("93506", na=False)]

# However, there are several more rows with zipcode 92440. Investigate
cleaned_further[cleaned_further["PostalCode"] == "92440"][["PostalCode", "Longitude", "Latitude",
                                                               "imputed_long", "imputed_lat", "City"]]

,PostalCode,Longitude,Latitude,imputed_long,imputed_lat,City
19419,92440,-116.567859,33.968449,-116.567859,33.968449,Desert Hot Springs
20731,92440,-116.566796,33.968075,-116.566796,33.968075,Desert Hot Springs
20733,92440,-116.566850,33.968494,-116.566850,33.968494,Desert Hot Springs
14116,92440,-116.565670,33.968423,-116.565670,33.968423,Desert Hot Springs
13461,92440,-116.565670,33.968423,-116.565670,33.968423,Desert Hot Springs
16179,92440,-116.565026,33.968733,-116.565026,33.968733,Desert Hot Springs
17110,92440,-116.566324,33.968378,-116.566324,33.968378,Desert Hot Springs
4453,92440,-116.564908,33.967852,-116.564908,33.967852,NaN
11701,92440,-116.564908,33.967852,-116.564908,33.967852,Desert Hot Springs
11707,92440,-116.565670,33.968423,-116.565670,33.968423,NaN


In [308]:
# These invalid zipcodes claim to be from the city Desert Hot Springs, CA.
# From a google search, the true zipcode of Desert Hot Springs, CA is actually 92240.

# Searching the MLS database for 92240 shows significantly more rows, also called Desert Hot Springs
cleaned_further[cleaned_further["PostalCode"] == "92240"][["PostalCode", "City"]]

# There must have been a data entry error that turned 92240 into 92440.

,PostalCode,City
3640,92240,Desert Hot Springs
4172,92240,Desert Hot Springs
4286,92240,Desert Hot Springs
4832,92240,Desert Hot Springs
4862,92240,Desert Hot Springs
...,...,...
21959,92240,Desert Hot Springs
22029,92240,Desert Hot Springs
22980,92240,Desert Hot Springs
23284,92240,Desert Hot Springs


In [309]:
# Impute missing city values of 92240 with Desert Hot Springs
cleaned_further.loc[(cleaned_further["City"].isna()) & 
                    (cleaned_further["PostalCode"] == "92440"),
                    "City"] = "Desert Hot Springs"

# Then change 92240 values to the correct 92240 zipcode
cleaned_further["PostalCode"] = cleaned_further["PostalCode"].replace("92440", "92240")

In [310]:
# Impute remaining city values with PostalCode-City dictionary

cleaned_further["imputed_city"] = cleaned_further["City"]
cleaned_further["imputed_city"] = cleaned_further["imputed_city"].fillna(cleaned_further["PostalCode"].map(city_dict))

In [311]:
# Drop the rest of the locational features for being redundant 

cleaned_further = cleaned_further.drop(columns=["Longitude", "Latitude", "UnparsedAddress", 
                                                "StreetNumberNumeric", "MLSAreaMajor", "City"])

In [312]:
# Keep HighSchoolDistrict for potential feature engineering later
# Fill its null values with unknown

cleaned_further["HighSchoolDistrict"] = cleaned_further["HighSchoolDistrict"].fillna("Unknown")

In [313]:
number_rows_dropped = prev[0] - cleaned_further.shape[0]
percent_rows_dropped = number_rows_dropped/prev[0] * 100

number_cols_dropped = prev[1] - cleaned_further.shape[1]
percent_cols_dropped = number_cols_dropped/prev[1] * 100

prev = cleaned_further.shape
print(prev)
print(f"Dropped {number_rows_dropped} rows, {round(percent_rows_dropped, 2)}%")
print(f"Dropped {number_cols_dropped} columns, {round(percent_cols_dropped, 2)}%")

(307655, 32)
Dropped 1 rows, 0.0%
Dropped 5 columns, 13.51%


### YearBuilt

In [314]:
# Impute YearBuilt with median YearBuilt from each city

cities = cleaned_further["imputed_city"].unique()

city_year_dict = {}

for city in cities:
    city_year_dict[city] = cleaned_further[cleaned_further["imputed_city"] == city]["YearBuilt"].median()

In [315]:
# Use dictionary of City-YearBuilt to fill missing YearBuilt values

cleaned_further["imputed_year_built"] = cleaned_further["YearBuilt"]
cleaned_further["imputed_year_built"] = cleaned_further["imputed_year_built"].fillna(
                                        cleaned_further["imputed_city"].map(city_year_dict))

In [316]:
# For any remaining missing values of YearBuilt, fill with median of YearBuilt

cleaned_further["imputed_year_built"] = cleaned_further["imputed_year_built"].fillna(
                                        cleaned_further["imputed_year_built"].median())

In [317]:
# Drop original YearBuilt column for being redundant

cleaned_further = cleaned_further.drop(columns = ["YearBuilt"])

### TestSetTrue

In [318]:
# TestSetTrue is a custom feature that just signifies which parts of the data are going to be used in the test set

cleaned_further["TestSetTrue"] = cleaned_further["TestSetTrue"].fillna(0)

-----------------------

In [319]:
number_rows_dropped = cleaned_leak.shape[0] - cleaned_further.shape[0]
percent_rows_dropped = number_rows_dropped/cleaned_leak.shape[0] * 100

number_cols_dropped = cleaned_leak.shape[1] - cleaned_further.shape[1]
percent_cols_dropped = number_cols_dropped/cleaned_leak.shape[1] * 100

prev = cleaned_further.shape
print(prev)
print(f"Dropped {number_rows_dropped} rows, {round(percent_rows_dropped, 2)}%")
print(f"Dropped {number_cols_dropped} columns, {round(percent_cols_dropped, 2)}%")

(307655, 32)
Dropped 1711 rows, 0.55%
Dropped 11 columns, 25.58%


In [320]:
cleaned_further.isna().sum()

Flooring                 0
ViewYN                   0
PoolPrivateYN            0
ListingKey               0
CloseDate                0
ClosePrice               0
PropertyType             0
LivingArea               0
ListOfficeName           0
ListingKeyNumeric        0
CountyOrParish           0
AttachedGarageYN         0
ParkingTotal             0
PropertySubType          0
ListingId                0
BathroomsTotalInteger    0
BedroomsTotal            0
StateOrProvince          0
FireplaceYN              0
MainLevelBedrooms        0
NewConstructionYN        0
GarageSpaces             0
HighSchoolDistrict       0
PostalCode               0
AssociationFee           0
LotSizeSquareFeet        0
TestSetTrue              0
imputed_stories          0
imputed_long             0
imputed_lat              0
imputed_city             0
imputed_year_built       0
dtype: int64

In [321]:
cleaned_further

,Flooring,ViewYN,PoolPrivateYN,ListingKey,CloseDate,ClosePrice,PropertyType,LivingArea,ListOfficeName,ListingKeyNumeric,...,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,TestSetTrue,imputed_stories,imputed_long,imputed_lat,imputed_city,imputed_year_built
5,Unknown,False,False,522107581,2024-01-05,815000.0,Residential,1974.0,Berkshire Hathaway HomeServices California Pro...,522107581,...,Unknown,91950,0.0,7020.0,0.0,2.0,-117.096922,32.659315,National City,2023.0
9,Unknown,False,False,510919001,2024-01-05,810000.0,Residential,1974.0,Berkshire Hathaway HomeServices California Pro...,510919001,...,Unknown,91950,0.0,7020.0,0.0,2.0,-117.097174,32.659284,National City,2023.0
28,Unknown,True,False,1067652762,2024-01-02,2100000.0,Residential,3736.0,Home & Ranch Sotheby's Intl,1067652762,...,San Luis Coastal Unified,93401,0.0,11219.0,0.0,2.0,-120.646786,35.277199,San Luis Obispo,2019.0
29,Tile,True,False,1063453216,2024-01-22,1950000.0,Residential,2100.0,Mendo Sotheby's International Realty,1063453216,...,Unknown,95437,0.0,74487.6,0.0,2.0,-123.814842,39.419080,Fort Bragg,1986.0
31,"Tile,Wood",True,Unknown,1061988701,2024-01-31,2340000.0,Residential,2442.0,Infiniti Real Estate,1061988701,...,Other,95148,0.0,8712.0,0.0,2.0,-121.780303,37.325454,San Jose,1985.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24227,"Carpet,Tile,Vinyl",True,False,1073242811,2026-04-30,525000.0,Residential,944.0,"LEGACY REALTY PARTNERS, INC",1073242811,...,Los Angeles Unified,91331,0.0,7971.0,1.0,1.0,-118.425478,34.265926,Pacoima,1947.0
24228,Unknown,False,False,1073214872,2026-04-01,405000.0,Residential,2176.0,"Sonya Carter, Broker",1073214872,...,Victor Valley Unified,92392,0.0,5200.0,1.0,2.0,-117.389973,34.474548,Victorville,1990.0
24229,Unknown,True,False,1069708148,2026-04-01,215000.0,Residential,1508.0,Century 21 Epic-Clearlake,1069708148,...,Konocti Unified,95424,0.0,10019.0,1.0,2.0,-122.694422,38.985590,Clearlake,1990.0
24231,"Carpet,Tile,Wood",False,False,1067010583,2026-04-13,415000.0,Residential,1583.0,ARROWHEAD PREMIER PROPERTIES,1067010583,...,Rim of the World,92322,0.0,12096.0,1.0,2.0,-117.324666,34.247895,Cedarpines Park,2010.0


## Handle Outliers

In [322]:
# Numerical features: 
# ClosePrice, LivingArea, LotSizeSquareFeet, AssociationFee,
# ParkingTotal, BathroomsTotalInteger, BedroomsTotal, MainLevelBedrooms, GarageSpaces, imputed_stories, 

In [323]:
# Log the large numeric values to shrink them down

cleaned_further = cleaned_further[cleaned_further["ClosePrice"] > 0]
cleaned_further["ClosePriceLog"] = np.log(cleaned_further["ClosePrice"])

cleaned_further = cleaned_further[cleaned_further["LivingArea"] > 0]
cleaned_further["LivingAreaLog"] = np.log(cleaned_further["LivingArea"])

cleaned_further = cleaned_further[cleaned_further["LotSizeSquareFeet"] > 0]
cleaned_further["LotSizeSquareFeetLog"] = np.log(cleaned_further["LotSizeSquareFeet"])

cleaned_further = cleaned_further[cleaned_further["AssociationFee"] >= 0]
cleaned_further["AssociationFeeLog"] = np.log(cleaned_further["AssociationFee"] + 1)

In [324]:
# For smaller numeric values, winsorize at 95%

upper_bound = cleaned_further["ParkingTotal"].quantile(0.95)
cleaned_further["ParkingTotalWinz"] = cleaned_further["ParkingTotal"].clip(lower=0, upper=upper_bound)

upper_bound = cleaned_further["BathroomsTotalInteger"].quantile(0.95)
cleaned_further["BathroomsTotalIntegerWinz"] = cleaned_further["BathroomsTotalInteger"].clip(lower=0, upper=upper_bound)

upper_bound = cleaned_further["BedroomsTotal"].quantile(0.95)
cleaned_further["BedroomsTotalWinz"] = cleaned_further["BedroomsTotal"].clip(lower=0, upper=upper_bound)

upper_bound = cleaned_further["MainLevelBedrooms"].quantile(0.95)
cleaned_further["MainLevelBedroomsWinz"] = cleaned_further["MainLevelBedrooms"].clip(lower=0, upper=upper_bound)

upper_bound = cleaned_further["GarageSpaces"].quantile(0.95)
cleaned_further["GarageSpacesWinz"] = cleaned_further["GarageSpaces"].clip(lower=0, upper=upper_bound)

upper_bound = cleaned_further["imputed_stories"].quantile(0.95)
cleaned_further["ImputedStoriesWinz"] = cleaned_further["imputed_stories"].clip(lower=0, upper=upper_bound)

## Encode Categorical Features

In [325]:
# Categorical features: 
# ListingKey, ListingKeyNumeric
# ViewYn, PoolPrivateYN, AttachedGarageYN, FireplaceYN, NewConstructionYN, 
# Flooring, PropertySubStype, StateOrProvince, HighSchoolDistrict, imputed_city, PropertyType, ListOfficeName, CountyOrParish

In [326]:
# Check if there's a difference between ListingKey and ListingKeyNumeric
# Since there is not, remove ListingKeyNumeric for being redundant

full = cleaned_further.shape[0]

matching = cleaned_further["ListingKey"].eq(cleaned_further["ListingKeyNumeric"]).sum()
print(f"Difference between ListingKey and ListingKeyNumeric: {full - matching}")

cleaned_further["ListingKey"] = cleaned_further["ListingKeyNumeric"]
cleaned_further = cleaned_further.drop(columns = ["ListingKeyNumeric"])

Difference between ListingKey and ListingKeyNumeric: 0


In [327]:
# Change Booleans to 0 = False, 1 = True, 0.5 = "Unknown"

cleaned_further["ViewYN"] = cleaned_further["ViewYN"].replace({False: 0, True: 1, "Unknown": 0.5})
cleaned_further["PoolPrivateYN"] = cleaned_further["PoolPrivateYN"].replace({False: 0, True: 1, "Unknown": 0.5})
cleaned_further["AttachedGarageYN"] = cleaned_further["AttachedGarageYN"].replace({False: 0, True: 1, "Unknown": 0.5})
cleaned_further["FireplaceYN"] = cleaned_further["FireplaceYN"].replace({False: 0, True: 1, "Unknown": 0.5})
cleaned_further["NewConstructionYN"] = cleaned_further["NewConstructionYN"].replace({False: 0, True: 1, "Unknown": 0.5})

/var/folders/p2/8wbydb310gg6z0sdxkw7vhym0000gn/T/ipykernel_93148/586058043.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_further["ViewYN"] = cleaned_further["ViewYN"].replace({False: 0, True: 1, "Unknown": 0.5})
/var/folders/p2/8wbydb310gg6z0sdxkw7vhym0000gn/T/ipykernel_93148/586058043.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_further["PoolPrivateYN"] = cleaned_further["PoolPrivateYN"].replace({False: 0, True: 1, "Unknown": 0.5})
/var/folders/p2/8wbydb310gg6z0sdxkw7vhym0000gn/T/ipykernel_93148/58605

In [328]:
# Check cardinality of remaining features

cleaned_further[["Flooring", "PropertyType", "ListOfficeName", "CountyOrParish", "PropertySubType",
                "StateOrProvince", "HighSchoolDistrict", "imputed_city"]].nunique()

Flooring                323
PropertyType              1
ListOfficeName        16649
CountyOrParish           63
PropertySubType           1
StateOrProvince          11
HighSchoolDistrict      444
imputed_city           1131
dtype: int64

In [329]:
# Previously filtered data set to PropertyType == Residential and PropertySubType == SingleFamilyResidence
# So these two features are useless, drop them

cleaned_further = cleaned_further.drop(columns=["PropertyType", "PropertySubType"])

### Low Cardinality Features

In [330]:
# Split the flooring values into a list

test = cleaned_further["Flooring"].str.split(",")
test

5                    [Unknown]
9                    [Unknown]
28                   [Unknown]
29                      [Tile]
31                [Tile, Wood]
                 ...          
24227    [Carpet, Tile, Vinyl]
24228                [Unknown]
24229                [Unknown]
24231     [Carpet, Tile, Wood]
24234                  [Vinyl]
Name: Flooring, Length: 307654, dtype: object

In [331]:
# One hot encode the the flooring list

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

encoded_df = pd.DataFrame(
    mlb.fit_transform(test),
    columns=mlb.classes_,
    index=cleaned_further.index)

# Add ListingKey and CloseDate to this new dataframe in order to match it back to orignial dataframe
encoded_df["ListingKey"] = cleaned_further["ListingKey"]
encoded_df["CloseDate"] = cleaned_further["CloseDate"]

encoded_df

,Bamboo,Brick,Carpet,Concrete,Laminate,SeeRemarks,Stone,Tile,Unknown,Vinyl,Wood,ListingKey,CloseDate
5,0,0,0,0,0,0,0,0,1,0,0,522107581,2024-01-05
9,0,0,0,0,0,0,0,0,1,0,0,510919001,2024-01-05
28,0,0,0,0,0,0,0,0,1,0,0,1067652762,2024-01-02
29,0,0,0,0,0,0,0,1,0,0,0,1063453216,2024-01-22
31,0,0,0,0,0,0,0,1,0,0,1,1061988701,2024-01-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24227,0,0,1,0,0,0,0,1,0,1,0,1073242811,2026-04-30
24228,0,0,0,0,0,0,0,0,1,0,0,1073214872,2026-04-01
24229,0,0,0,0,0,0,0,0,1,0,0,1069708148,2026-04-01
24231,0,0,1,0,0,0,0,1,0,0,1,1067010583,2026-04-13


In [332]:
# Merge the one-hot-encoded flooring columns back to original dataset

cleaned_further = pd.merge(cleaned_further, encoded_df, on=["ListingKey", "CloseDate"]).drop(columns=["CloseDate", "Flooring"])
cleaned_further.head()

,ViewYN,PoolPrivateYN,ListingKey,ClosePrice,LivingArea,ListOfficeName,CountyOrParish,AttachedGarageYN,ParkingTotal,ListingId,...,Brick,Carpet,Concrete,Laminate,SeeRemarks,Stone,Tile,Unknown,Vinyl,Wood
0,0.0,0.0,522107581,815000.0,1974.0,Berkshire Hathaway HomeServices California Pro...,San Diego,1.0,2.0,210014555,...,0,0,0,0,0,0,0,1,0,0
1,0.0,0.0,510919001,810000.0,1974.0,Berkshire Hathaway HomeServices California Pro...,San Diego,1.0,2.0,210008330,...,0,0,0,0,0,0,0,1,0,0
2,1.0,0.0,1067652762,2100000.0,3736.0,Home & Ranch Sotheby's Intl,San Luis Obispo,1.0,3.0,NS24001417,...,0,0,0,0,0,0,0,1,0,0
3,1.0,0.0,1063453216,1950000.0,2100.0,Mendo Sotheby's International Realty,Mendocino,1.0,2.0,C1-10456,...,0,0,0,0,0,0,1,0,0,0
4,1.0,0.5,1061988701,2340000.0,2442.0,Infiniti Real Estate,Santa Clara,1.0,3.0,ML81956298,...,0,0,0,0,0,0,1,0,0,1


In [333]:
# One hot encode the state feaure

cleaned_further = pd.get_dummies(cleaned_further, columns=["StateOrProvince"], dtype=int)

### High Cardinality Features

In [334]:
pip install category_encoders

Note: you may need to restart the kernel to use updated packages.


In [335]:
# Use target encoding for: 
# ListOfficeName, CountyOrParish, HighSchoolDistrict, imputed_city, PostalCode, imputed_year_built

import category_encoders as ce

encoder = ce.TargetEncoder(cols=["ListOfficeName"])
list_office_enc = encoder.fit_transform(cleaned_further["ListOfficeName"], cleaned_further["ClosePrice"])
cleaned_further["ListOfficeNameEnc"] = list_office_enc

encoder = ce.TargetEncoder(cols=["CountyOrParish"])
county_enc = encoder.fit_transform(cleaned_further["CountyOrParish"], cleaned_further["ClosePrice"])
cleaned_further["CountyOrParishEnc"] = county_enc

encoder = ce.TargetEncoder(cols=["HighSchoolDistrict"])
highschool_enc = encoder.fit_transform(cleaned_further["HighSchoolDistrict"], cleaned_further["ClosePrice"])
cleaned_further["HighSchoolDistrictEnc"] = highschool_enc

encoder = ce.TargetEncoder(cols=["imputed_city"])
city_enc = encoder.fit_transform(cleaned_further["imputed_city"], cleaned_further["ClosePrice"])
cleaned_further["imputed_city_enc"] = city_enc

encoder = ce.TargetEncoder(cols=["PostalCode"])
zip_enc = encoder.fit_transform(cleaned_further["PostalCode"], cleaned_further["ClosePrice"])
cleaned_further["PostalCodeEnc"] = zip_enc

encoder = ce.TargetEncoder(cols=["imputed_year_built"])
year_enc = encoder.fit_transform(cleaned_further["imputed_year_built"], cleaned_further["ClosePrice"])
cleaned_further["imputed_year_built_enc"] = year_enc

In [336]:
cleaned_further.columns

Index(['ViewYN', 'PoolPrivateYN', 'ListingKey', 'ClosePrice', 'LivingArea',
       'ListOfficeName', 'CountyOrParish', 'AttachedGarageYN', 'ParkingTotal',
       'ListingId', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN',
       'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'TestSetTrue', 'imputed_stories', 'imputed_long',
       'imputed_lat', 'imputed_city', 'imputed_year_built', 'ClosePriceLog',
       'LivingAreaLog', 'LotSizeSquareFeetLog', 'AssociationFeeLog',
       'ParkingTotalWinz', 'BathroomsTotalIntegerWinz', 'BedroomsTotalWinz',
       'MainLevelBedroomsWinz', 'GarageSpacesWinz', 'ImputedStoriesWinz',
       'Bamboo', 'Brick', 'Carpet', 'Concrete', 'Laminate', 'SeeRemarks',
       'Stone', 'Tile', 'Unknown', 'Vinyl', 'Wood', 'StateOrProvince_AZ',
       'StateOrProvince_CA', 'StateOrProvince_CO', 'StateOrProvince_FL',
       'StateOrProvince_GA', 'StateOrProvince_

In [337]:
cleaned_further.shape

(307654, 64)

## Train/Test Split

In [338]:
train_set = cleaned_further[cleaned_further["TestSetTrue"] == 0]
test_set = cleaned_further[cleaned_further["TestSetTrue"] == 1]

X_train = train_set[["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
                    "LivingAreaLog", "LotSizeSquareFeetLog", "AssociationFeeLog",
                    "ParkingTotalWinz", 'BathroomsTotalIntegerWinz', 'BedroomsTotalWinz', 'MainLevelBedroomsWinz','GarageSpacesWinz', 'ImputedStoriesWinz',
                    'Bamboo', 'Brick', 'Carpet', 'Concrete', 'Laminate', 'SeeRemarks', 'Stone', 'Tile', 'Unknown', 'Vinyl', 'Wood',
                    'StateOrProvince_AZ','StateOrProvince_CA', 'StateOrProvince_CO', 'StateOrProvince_FL', 'StateOrProvince_GA',
                    'StateOrProvince_ME', 'StateOrProvince_NV','StateOrProvince_NY', 'StateOrProvince_OS', 'StateOrProvince_TN', 'StateOrProvince_TX',
                    'ListOfficeNameEnc','CountyOrParishEnc', 'HighSchoolDistrictEnc', 'imputed_city_enc', 'PostalCodeEnc', 'imputed_year_built_enc',
                    "imputed_long", "imputed_lat"]]
X_test = test_set[["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
                    "LivingAreaLog", "LotSizeSquareFeetLog", "AssociationFeeLog",
                    "ParkingTotalWinz", 'BathroomsTotalIntegerWinz', 'BedroomsTotalWinz', 'MainLevelBedroomsWinz','GarageSpacesWinz', 'ImputedStoriesWinz',
                    'Bamboo', 'Brick', 'Carpet', 'Concrete', 'Laminate', 'SeeRemarks', 'Stone', 'Tile', 'Unknown', 'Vinyl', 'Wood',
                    'StateOrProvince_AZ','StateOrProvince_CA', 'StateOrProvince_CO', 'StateOrProvince_FL', 'StateOrProvince_GA',
                    'StateOrProvince_ME', 'StateOrProvince_NV','StateOrProvince_NY', 'StateOrProvince_OS', 'StateOrProvince_TN', 'StateOrProvince_TX',
                    'ListOfficeNameEnc','CountyOrParishEnc', 'HighSchoolDistrictEnc', 'imputed_city_enc', 'PostalCodeEnc', 'imputed_year_built_enc',
                    "imputed_long", "imputed_lat"]]

y_train = train_set["ClosePriceLog"]
y_test = test_set["ClosePriceLog"]

In [339]:
cleaned_further.to_csv("week_3_cleaned_op2.csv", index=False)

## Summary

### Dataset Shape Per Stage

* Orignial dataset shape: (309735, 85)
    - No changes

* After removing impossibilites: (309366, 85)
    - Dropped 369 rows, 0.12%
    - Dropped 0 columns, 0.00%

* Afer dropping columns with majority null values: (309366, 56)
    - Dropped 0 rows, 0.00%
    - Dropped 29 columns, 34.12%

* After removing leakage: (309366, 43)
    - Dropped 0 rows, 0.00%
    - Dropped 13 columns, 23.21%

* After handling remaining missing values: (307655, 32)
    - Dropped 1711 rows, 0.55%
    - Dropped 11 columns, 25.58%

* Final Cleaned Dataset: (307654, 64)

| Dropped | No Change | Imputed | Filled with "Unknown" |
| -------- | -------- | -------- | --------
| FireplacesTotal | ListingKey | Stories (imputed_stories) | Flooring
| CoveredSpaces | StateOrProvince | BathroomsTotalInteger | ViewYN
| AboveGradeFinishedArea | BedroomsTotal | AssociationFee | PoolPrivateYN
| TaxAnnualAmount| ListingId | LotSizeSquareFeet | AttachedGarageYN
| TaxYear | CountyOrParish | MainLevelBedrooms | FireplaceYN
| ElementarySchoolDistrict | PostalCode | YearBuilt (imputed_year_built) | NewConstructionYN
| BusinessType | ListOfficeName | LivingArea | HighSchoolDistrict
| MiddleOrJuniorSchoolDistrict | CloseDate | ParkingTotal | 
| WaterfrontYN | ClosePrice | GarageSpaces | 
| BelowGradeFinishedArea | | Longitude (imputed_long) | 
| BasementYN | | Latitude (imputed_lat) | 
| BuilderName |  | City (imputed_city) | 
| LotSizeDimensions | |  | 
| BuildingAreaTotal | 
| CoBuyerAgentFirstName | 
| OriginatingSystemName | 
| OriginatingSystemSubName | 
| BuyerAgencyCompensation | 
| BuyerAgencyCompensationType | 
| ElementarySchool | 
| MiddleOrJuniorSchool | 
| HighSchool | 
| CoListAgentFirstName | 
| CoListAgentLastName | 
| CoListOfficeName | 
| AssociationFeeFrequency
| lonfilled
| latfilled
| SubdivisionName
| OriginalListPrice
| ListPrice
| DaysOnMarket
| BuyerOfficeName
| BuyerAgentMlsId
| BuyerAgentFirstName
| BuyerAgentLastName
| MlsStatus
| BuyerOfficeAOR
| PurchaseContractDate
| BuyerAgentAOR
| ContractStatusChangeDate
| ListingContractDate
| ListAgentFirstName
| ListAgentLastName
| ListAgentAOR
| ListAgentEmail
| ListAgentFullName
| Levels
| LotSizeAcres
| LotSizeArea
| UnparsedAddress
| StreetNumberNumeric
| MLSAreaMajor
| ListingKeyNumeric
| PropertyType
| PropertySubType
                                    